# MRI Reconstruction Training — Kaggle (4x)

Train U-Net for **4x** accelerated MRI reconstruction.  
All code is inlined — no external `src/` dependency needed.

**Steps:**
1. Add fastMRI dataset (ensure path below matches)
2. Enable GPU accelerator (P100)
3. Run all cells
4. Download `best_4x.pt` from `/kaggle/working/checkpoints/`

In [ ]:
# Cell 1: Setup + verify environment
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

import os
DATA_ROOT = '/kaggle/input/fastmri/fastmri_pd'
print(f'\nDataset: {DATA_ROOT}')
print(f'Train files: {len(os.listdir(os.path.join(DATA_ROOT, "train", "h5")))}')
print(f'Val files: {len(os.listdir(os.path.join(DATA_ROOT, "val", "h5")))}')

In [ ]:
# Cell 2: All source code (masks, data, model, losses, metrics, training)

from dataclasses import dataclass
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset

# =====================================================================
# Masks
# =====================================================================

def create_cartesian_mask(width, acceleration, center_fraction, rng):
    mask = np.zeros(width, dtype=bool)
    num_center = int(center_fraction * width)
    center_start = (width - num_center) // 2
    mask[center_start : center_start + num_center] = True
    total_lines = width // acceleration
    remaining = total_lines - num_center
    if remaining > 0:
        available = np.where(~mask)[0]
        chosen = rng.choice(available, size=remaining, replace=False)
        mask[chosen] = True
    return mask

# =====================================================================
# FFT utilities
# =====================================================================

def fft2c(image):
    return torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(image)))

def ifft2c(kspace):
    return torch.fft.fftshift(torch.fft.ifft2(torch.fft.ifftshift(kspace)))

# =====================================================================
# Dataset
# =====================================================================

class FastMRIDataset(Dataset):
    def __init__(self, file_paths, acceleration, center_fraction, seed=42, augment=False):
        self.file_paths = sorted(file_paths)
        self.acceleration = acceleration
        self.center_fraction = center_fraction
        self.seed = seed
        self.augment = augment
        self._epoch = 0

    def set_epoch(self, epoch):
        self._epoch = epoch

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        with h5py.File(self.file_paths[idx], "r") as f:
            target_np = np.array(f["image_rss"][:], dtype=np.float32)

        mean = float(target_np.mean())
        std = float(target_np.std()) + 1e-11
        target_norm = (target_np - mean) / std

        target_t = torch.from_numpy(target_norm).unsqueeze(0)
        kspace = fft2c(target_t)

        # Stochastic masks during training, deterministic during eval
        if self.augment:
            rng = np.random.default_rng(self.seed + idx + self._epoch * len(self))
        else:
            rng = np.random.default_rng(self.seed + idx)

        mask_np = create_cartesian_mask(
            target_np.shape[1], self.acceleration, self.center_fraction, rng,
        )
        mask_t = torch.from_numpy(mask_np).unsqueeze(0)

        undersampled_kspace = kspace * mask_t.unsqueeze(1)
        zf_image = torch.abs(ifft2c(undersampled_kspace))

        # Random flips during training
        if self.augment and rng.random() > 0.5:
            target_t = torch.flip(target_t, [-1])
            zf_image = torch.flip(zf_image, [-1])
            undersampled_kspace = fft2c(target_t) * mask_t.unsqueeze(1)

        return {
            "input": zf_image.float(),
            "target": target_t.float(),
            "mask": mask_t,
            "kspace": undersampled_kspace,
            "mean": mean,
            "std": std,
            "fname": self.file_paths[idx].name,
        }

def get_file_paths(h5_dir):
    return sorted(Path(h5_dir).glob("*.h5"))

# =====================================================================
# U-Net
# =====================================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout_rate=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        self.dropout = nn.Dropout2d(p=dropout_rate) if dropout_rate > 0 else nn.Identity()

    def forward(self, x):
        return self.dropout(self.conv(x))

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=(32, 64, 128, 256), dropout_rate=0.0):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools = nn.ModuleList()
        prev_ch = in_channels
        for feat in features:
            self.encoders.append(ConvBlock(prev_ch, feat, dropout_rate))
            self.pools.append(nn.MaxPool2d(2, 2))
            prev_ch = feat

        self.bottleneck = ConvBlock(features[-1], features[-1] * 2, dropout_rate)

        self.upconvs = nn.ModuleList()
        self.decoders = nn.ModuleList()
        for feat in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(feat * 2, feat, 2, 2))
            self.decoders.append(ConvBlock(feat * 2, feat, dropout_rate))

        self.final_conv = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x)
            skips.append(x)
            x = pool(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.upconvs, self.decoders, reversed(skips)):
            x = up(x)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
        return self.final_conv(x)

    def enable_mc_dropout(self):
        for m in self.modules():
            if isinstance(m, nn.Dropout2d):
                m.train()

# =====================================================================
# Losses
# =====================================================================

def _gaussian_window(size, sigma=1.5):
    coords = torch.arange(size, dtype=torch.float32) - size // 2
    g = torch.exp(-(coords**2) / (2 * sigma**2))
    return g / g.sum()

def _create_window(size, channels):
    w1d = _gaussian_window(size)
    w2d = w1d.unsqueeze(1) * w1d.unsqueeze(0)
    return w2d.unsqueeze(0).unsqueeze(0).expand(channels, 1, size, size).contiguous()

def compute_ssim(pred, target, window_size=7, data_range=1.0):
    channels = pred.shape[1]
    window = _create_window(window_size, channels).to(pred.device, pred.dtype)
    pad = window_size // 2
    mu_p = F.conv2d(pred, window, padding=pad, groups=channels)
    mu_t = F.conv2d(target, window, padding=pad, groups=channels)
    sigma_pp = F.conv2d(pred**2, window, padding=pad, groups=channels) - mu_p**2
    sigma_tt = F.conv2d(target**2, window, padding=pad, groups=channels) - mu_t**2
    sigma_pt = F.conv2d(pred * target, window, padding=pad, groups=channels) - mu_p * mu_t
    c1 = (0.01 * data_range) ** 2
    c2 = (0.03 * data_range) ** 2
    ssim_map = ((2 * mu_p * mu_t + c1) * (2 * sigma_pt + c2)) / (
        (mu_p**2 + mu_t**2 + c1) * (sigma_pp + sigma_tt + c2)
    )
    return ssim_map.mean()

class ReconLoss(nn.Module):
    def __init__(self, ssim_weight=0.5):
        super().__init__()
        self.ssim_weight = ssim_weight

    def forward(self, pred, target):
        l1 = F.l1_loss(pred, target)
        ssim_val = compute_ssim(pred, target)
        return (1 - self.ssim_weight) * l1 + self.ssim_weight * (1 - ssim_val)

# =====================================================================
# Metrics
# =====================================================================

def psnr(pred, target, data_range=1.0):
    mse = torch.mean((pred - target) ** 2).item()
    if mse < 1e-10:
        return float("inf")
    return float(10 * torch.log10(torch.tensor(data_range**2 / mse)).item())

def ssim_metric(pred, target, data_range=1.0):
    return float(compute_ssim(pred, target, data_range=data_range).item())

# =====================================================================
# Training loop
# =====================================================================

@dataclass(frozen=True)
class TrainConfig:
    data_root: Path = Path("dataset/fastmri_pd")
    checkpoint_dir: Path = Path("outputs/checkpoints")
    acceleration: int = 4
    center_fraction: float = 0.08
    seed: int = 42
    features: tuple = (32, 64, 128, 256)
    dropout_rate: float = 0.05
    batch_size: int = 16
    num_epochs: int = 30
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    ssim_weight: float = 0.5
    num_workers: int = 2
    device: str = "cuda"

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_psnr, n = 0.0, 0.0, 0
    for batch in loader:
        inputs = batch["input"].to(device)
        targets = batch["target"].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        data_range = float((targets.max() - targets.min()).item())
        total_loss += loss.item()
        total_psnr += psnr(outputs.detach(), targets, data_range=data_range)
        n += 1
    return {"loss": total_loss / n, "psnr": total_psnr / n}

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, total_psnr, total_ssim, n = 0.0, 0.0, 0.0, 0
    for batch in loader:
        inputs = batch["input"].to(device)
        targets = batch["target"].to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        data_range = float((targets.max() - targets.min()).item())
        total_loss += loss.item()
        total_psnr += psnr(outputs, targets, data_range=data_range)
        total_ssim += ssim_metric(outputs, targets, data_range=data_range)
        n += 1
    return {"loss": total_loss / n, "psnr": total_psnr / n, "ssim": total_ssim / n}

def train(config):
    device = torch.device(config.device)
    print(f"Training on {device} | {config.acceleration}x acceleration")

    train_paths = get_file_paths(config.data_root / "train" / "h5")
    val_paths = get_file_paths(config.data_root / "val" / "h5")

    train_ds = FastMRIDataset(train_paths, config.acceleration, config.center_fraction,
                               config.seed, augment=True)
    val_ds = FastMRIDataset(val_paths, config.acceleration, config.center_fraction, config.seed)
    train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,
                              num_workers=config.num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False,
                            num_workers=config.num_workers, pin_memory=True)
    print(f"Train: {len(train_ds)} samples | Val: {len(val_ds)} samples")

    model = UNet(features=config.features, dropout_rate=config.dropout_rate).to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate,
                                   weight_decay=config.weight_decay)
    criterion = ReconLoss(ssim_weight=config.ssim_weight)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)

    config.checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_path = config.checkpoint_dir / f"best_{config.acceleration}x.pt"
    best_ssim = 0.0

    for epoch in range(1, config.num_epochs + 1):
        train_loader.dataset.set_epoch(epoch)
        t = train_one_epoch(model, train_loader, optimizer, criterion, device)
        v = validate(model, val_loader, criterion, device)
        scheduler.step(v["ssim"])
        lr = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch:02d}/{config.num_epochs} | "
              f"Train Loss: {t['loss']:.4f} | Val PSNR: {v['psnr']:.2f} dB | "
              f"Val SSIM: {v['ssim']:.4f} | LR: {lr:.1e}")

        if v["ssim"] > best_ssim:
            best_ssim = v["ssim"]
            torch.save({
                "model_state_dict": model.state_dict(),
                "config": {
                    "acceleration": config.acceleration,
                    "features": config.features,
                    "dropout_rate": config.dropout_rate,
                },
                "epoch": epoch,
                "val_ssim": best_ssim,
                "val_psnr": v["psnr"],
            }, best_path)
            print(f"  -> Saved best model (SSIM: {best_ssim:.4f})")

    print(f"Training complete. Best SSIM: {best_ssim:.4f}")
    return best_path

print("All code loaded.")

In [ ]:
# Cell 3: Train 4x model
config = TrainConfig(
    data_root=Path('/kaggle/input/fastmri/fastmri_pd'),
    checkpoint_dir=Path('/kaggle/working/checkpoints'),
    acceleration=4,
    device='cuda',
    num_workers=2,
)

best = train(config)
print(f'\nModel saved: {best}')

In [ ]:
# Cell 5: Verify + zip checkpoints for download
import shutil
checkpoint_dir = Path('/kaggle/working/checkpoints')
for f in sorted(checkpoint_dir.glob('*.pt')):
    size_mb = f.stat().st_size / 1e6
    ckpt = torch.load(f, map_location='cpu', weights_only=True)
    print(f'{f.name}: {size_mb:.1f} MB | Epoch {ckpt["epoch"]} | '
          f'SSIM {ckpt["val_ssim"]:.4f} | PSNR {ckpt["val_psnr"]:.2f} dB')

shutil.make_archive('/kaggle/working/checkpoints', 'zip', checkpoint_dir)
print(f'\nDownload: /kaggle/working/checkpoints.zip ({Path("/kaggle/working/checkpoints.zip").stat().st_size / 1e6:.1f} MB)')